# 考古研究日志

**目的**: 记录考古 AI/ML 领域的调研发现，包括数据集、工具、方法论。
**成功标准**: 整理出对我们项目（墓葬分析）有实际参考价值的信息。

In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
from datetime import datetime

print(f"Log entry: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 调研：考古 AI/ML 数据集现状

**核心发现**: 考古领域**没有公认的"标准大数据集"**——这是领域的现实困境。

数据极度碎片化，各团队自建数据集，没有统一标准。以下是我们调研到的信息。

In [ ]:
# 考古 CV 论文中实际引用最多的通用预训练底座

archaeology_pretrained = {
    "ImageNet": {
        "用途": "几乎所有考古 CV 论文都用它做迁移学习的预训练权重",
        "类型": "通用图像分类数据集",
        "规模": "~1400万图片, 1000类",
        "考古场景": "器物分类、墓葬图像识别、遗址照片分析",
    },
    "COCO": {
        "用途": "目标检测任务的预训练底座",
        "引用率": "约 41% 的考古结构检测研究用到",
        "类型": "通用目标检测/分割数据集",
        "规模": "~33万图片, 80类",
        "考古场景": "遗址结构检测、器物定位、航拍目标识别",
    },
}

for name, info in archaeology_pretrained.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in info.items():
        print(f"  {k}: {v}")

In [ ]:
# 考古专用数据集（相对较知名）

archaeology_datasets = {
    "Archaeoscape": {
        "发布": "2024, NeurIPS",
        "类型": "LiDAR 遗址检测",
        "特点": "目前学术引用最新、最正规的考古专用数据集",
        "适用": "地表遗址识别、遥感考古",
        "与我们项目关系": "不直接相关（我们做墓葬分析，非遥感），但标注规范值得参考",
    },
    "Kaggle OpenAI to Z Challenge": {
        "发布": "2024-2025",
        "类型": "考古 AI 竞赛数据",
        "特点": "今年热度最高的考古 AI 竞赛",
        "适用": "遗址发现、卫星图像分析",
        "与我们项目关系": "竞赛思路可参考，但数据类型不同",
    },
}

for name, info in archaeology_datasets.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in info.items():
        print(f"  {k}: {v}")

In [ ]:
# 对我们项目的实际策略

strategy = {
    "第一步": {
        "动作": "用 ImageNet/COCO 预训练模型做迁移学习",
        "原因": "考古领域普遍采用，不需要自己从头训练",
        "具体": "PyTorch torchvision.models 预训练权重",
    },
    "第二步": {
        "动作": "自己标注小规模领域数据",
        "工具推荐": "Roboflow（在线标注平台，支持导出多种格式）",
        "规模": "100-500 张图片即可开始微调",
        "标注内容": "墓葬形制、器物类型、纹饰类别",
    },
    "第三步": {
        "动作": "参考 Archaeoscape 的标注规范",
        "原因": "2024年最新、最正规的考古数据集标注标准",
        "适用": "统一我们的标注格式，便于后续扩展",
    },
}

print("项目数据策略路线图")
print("=" * 60)
for step, info in strategy.items():
    print(f"\n{step}: {info['动作']}")
    for k, v in info.items():
        if k != '动作':
            print(f"  {k}: {v}")

## 结论

- **考古 ML 领域数据集极度碎片化**，不存在像 ImageNet 那样的标准 benchmark
- **实际做法**: 几乎所有团队都用 ImageNet/COCO 预训练 + 自建小规模数据集
- **对我们的启示**: 不要等待"完美数据集"，先用手头数据 + 迁移学习做起来
- **Roboflow** 可以作为标注工具，降低标注门槛
- **Archaeoscape 的标注规范** 可以作为参考标准

---

## 行动计划：下一步（2026-07-10）

以下是按优先级排列的下一步行动。每完成一项打勾，有突破就在此 notebook 记录。

### 第一层：环境与基础设施（今天就做）

| # | 任务 | 说明 | 阻塞项 | 状态 |
|---|------|------|--------|------|
| 1 | VPS 安装 Python 工具链 | `uv` + `ruff` + `pyright`，用 `modern-python-toolchain` skill | 无 | ⬜ |
| 2 | VPS 安装 QGIS | `apt install qgis`，确认版本 ≥ 3.x | 无 | ⬜ |
| 3 | VPS 安装项目依赖 | `uv pip install torch torchvision pandas geopandas matplotlib` | #1 | ⬜ |
| 4 | 本地同步项目到 VPS | `git push` 本地改动 → VPS `git pull` | 无 | ⬜ |

**成功标准**: VPS 上能运行 `python -c "import torch; import geopandas; import qgis"` 不报错。

### 第二层：PDF 数据提取（等用户提供报告后立即执行）

| # | 任务 | 说明 | 阻塞项 | 状态 |
|---|------|------|--------|------|
| 5 | 读取 PDF 报告 | 用 `pdf` skill 或 `pymupdf` 提取文本 | 用户提供 PDF | ⬜ |
| 6 | 提取墓葬结构化数据 | 按 `CLAUDE.md` 中的考古信息提取规范，输出 JSON | #5 | ⬜ |
| 7 | 数据质量校验 | 检查墓口长/宽/深数值合理性，器物编号完整性 | #6 | ⬜ |
| 8 | 保存为 CSV/JSON | 存入 `data/` 目录，供后续分析使用 | #7 | ⬜ |

**成功标准**: 从至少一份 PDF 中提取出结构化的墓葬数据（含墓葬编号、尺寸、器物清单）。

**提取字段清单**（参考 `CLAUDE.md`）:
- 墓葬基本信息：墓葬编号、年代、墓向、墓葬形制、墓口长/宽、墓深、备注
- 随葬器物：器物编号、器物名称、材质、器型、数量、特征描述

### 第三层：QGIS 可视化（数据到手后执行）

| # | 任务 | 说明 | 阻塞项 | 状态 |
|---|------|------|--------|------|
| 9 | 创建 QGIS 项目模板 | 设置 CRS（WGS84 / CGCS2000）、底图、图层结构 | #2 | ⬜ |
| 10 | 墓葬点位图 | 将提取的墓葬坐标导入为点图层，按年代/形制着色 | #8, #9 | ⬜ |
| 11 | 分布热力图 | 按年代分层制作墓葬分布图，观察空间聚集 | #10 | ⬜ |
| 12 | 器物类型空间分布 | 不同器物类型在空间上的分布差异 | #10 | ⬜ |

**成功标准**: 在 QGIS 中看到墓葬的空间分布图，能初步判断是否存在区域聚集模式。

**关键问题**（数据说话后再回答）:
- 不同时期墓葬的空间分布是否有明显偏移？
- 特定器物类型是否集中在某些区域？
- 墓葬形制的地理分布是否有规律？

### 第四层：PyTorch 图像分类（可并行准备）

| # | 任务 | 说明 | 阻塞项 | 状态 |
|---|------|------|--------|------|
| 13 | 下载 ArchaeoMind 数据集 | Kaggle, 1349 张考古器物图片 | 无（可先做） | ⬜ |
| 14 | 下载 Persian Pottery 数据集 | GitHub, 660 张波斯陶器图片 | 无（可先做） | ⬜ |
| 15 | 搭建 PyTorch 分类脚本 | ResNet18 预训练 + 迁移学习 scaffold | #3, #13 | ⬜ |
| 16 | 在 ArchaeoMind 上测试 | 跑通训练-评估流程，确认 pipeline 可用 | #15 | ⬜ |

**成功标准**: 用预训练 ResNet18 在考古器物数据集上达到 >60% 分类准确率（baseline）。

**技术栈**:
- 预训练底座: `torchvision.models.resnet18(weights='IMAGENET1K_V1')`
- 数据增强: RandomHorizontalFlip, ColorJitter, RandomRotation
- 评估: confusion matrix + per-class accuracy

### 第五层：Baidu OCR 集成（可并行）

| # | 任务 | 说明 | 阻塞项 | 状态 |
|---|------|------|--------|------|
| 17 | 验证 Baidu OCR API 连通性 | 运行 `scripts/test_baidu_ocr.py`，确认 API Key 有效 | 无 | ⬜ |
| 18 | 构建 PDF→OCR→结构化 pipeline | PDF 图片 → Baidu OCR → 文本 → 考古信息提取 | #17 | ⬜ |
| 19 | 对比 OCR vs 直接 PDF 文本提取 | 哪种方式提取质量更高？ | #18 | ⬜ |

**成功标准**: 能用 Baidu OCR 从扫描版 PDF 中提取文字，并正确识别墓葬编号和尺寸。

### 并行执行策略

以下任务**互不依赖，可以同时进行**：

```
第一层（环境） ──→ 第二层（PDF提取） ──→ 第三层（QGIS）
                  ↗
第四层（数据集下载 + PyTorch scaffold） ← 不依赖 PDF
第五层（Baidu OCR） ← 不依赖其他层
```

**今天可以先做的**:
1. VPS 环境搭建（第一层）
2. 下载 ArchaeoMind + Persian Pottery 数据集（第四层）
3. 验证 Baidu OCR 连通性（第五层）

**等用户输入才能做的**:
- PDF 报告 → 才能做第二层 → 才能做第三层

### 突破记录

以下记录每个关键突破的时间和发现。完成一项就在这里加一条。

In [ ]:
# 突破记录模板
breakthroughs = [
    {
        "时间": "2026-07-10",
        "突破": "VPS SSH key 配置完成，项目在 VPS 上成功 clone",
        "细节": "生成 ed25519 key (vps-kaogu)，添加到 GitHub account alexxhchen-nu",
    },
    {
        "时间": "2026-07-10",
        "突破": "考古 ML 数据集调研完成",
        "细节": "ArchaeoMind (Kaggle), Persian Pottery (GitHub), Archaeoscape (NeurIPS 2024) 已记录在 dataset-research.md",
    },
    {
        "时间": "2026-07-10",
        "突破": "行动计划拆解为 5 层 19 项任务",
        "细节": "环境→PDF→QGIS→PyTorch→OCR，可并行执行",
    },
]

for b in breakthroughs:
    print(f"[{b['时间']}] {b['突破']}")
    print(f"  细节: {b['细节']}\n")